[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mento-calc/mento/blob/main/docs/source/examples/viga_rectangular_diseno_CIRSOC_201-25.ipynb)

In [ ]:
# Google Colab no trae mento instalado, así que lo instalamos sólo cuando el
# notebook corre allí. La guarda evita que la documentación llame a pip.
import sys

if "google.colab" in sys.modules:
    %pip install mento

# Diseño de viga rectangular — CIRSOC 201-25

Este ejemplo dimensiona la armadura de una viga rectangular de hormigón armado según el
**CIRSOC 201-2025**, el reglamento argentino. mento la implementa en `Concrete_CIRSOC_201_25`,
que comparte las expresiones del ACI 318-19 y trabaja siempre en unidades métricas.

*Diseñar* significa que mento elige la armadura: le damos la sección, los materiales y los
estados de carga, y devuelve las barras longitudinales y los estribos necesarios. Si en
cambio ya tenés la armadura y querés verificarla, mirá el ejemplo de
[verificación](viga_rectangular_verificacion_CIRSOC_201-25.ipynb).


#### Idioma español para salida de anexos

In [ ]:
import mento

mento.set_language("es")

**Materiales y geometría de la sección**

In [ ]:
from mento import Concrete_CIRSOC_201_25, SteelBar, RectangularBeam, cm, kN, MPa, kNm
from mento import Forces, Node

# Materiales
hormigon = Concrete_CIRSOC_201_25(name="H-25", f_c=25 * MPa)
acero = SteelBar(name="ADN 420", f_y=420 * MPa)

# Geometría de la viga: ancho, altura y recubrimiento
viga = RectangularBeam(
    label="V101",
    concrete=hormigon,
    steel_bar=acero,
    width=20 * cm,
    height=50 * cm,
    c_c=2.5 * cm,
)
viga.data

**Estados de carga**

Cada `Forces` es una combinación mayorada. `M_y` es el momento flector y `V_z` el esfuerzo de
corte; el signo del momento distingue la cara traccionada (positivo tracciona abajo).

In [ ]:
f1 = Forces(label="1.2D+1.6L", M_y=80 * kNm, V_z=70 * kN)
f2 = Forces(label="1.2D+1.6L+W", M_y=-35 * kNm, V_z=45 * kN)
print(f1)
print(f2)

**Nodo: sección más estados de carga**

El `Node` une la sección con la lista de combinaciones. Es sobre el nodo donde se corre el
diseño, y mento dimensiona para la combinación más exigente de la lista.

In [ ]:
nodo = Node(section=viga, forces=[f1, f2])
nodo

**Diseño a flexión y corte**

In [ ]:
# Dimensiona armadura longitudinal y estribos
nodo.design()

# Resumen de resultados
nodo.results

In [ ]:
# Sección con la armadura adoptada
viga.plot()

**Resultados en tablas**

`check_flexure()` y `check_shear()` devuelven un `DataFrame` de pandas con una fila por
combinación, cómodo para revisar varias vigas o exportar a Excel.

In [ ]:
nodo.check_flexure()

In [ ]:
nodo.check_shear()

**Leer la armadura adoptada desde el código**

Cuando lo que necesitás no es la tabla sino los valores — por ejemplo para armar una planilla
de despiece o alimentar otro cálculo — usá `viga.flexure_design` y `viga.shear_design`. Son
objetos de datos, con unidades, que devuelven exactamente lo que el diseño adoptó.

Cada cantidad es un `Quantity` de pint, así que se convierte a la unidad que prefieras con
`.to()`. Las áreas requeridas y los DCR son la envolvente de todos los estados de carga: el
DCR de una cara es el de la combinación que la gobierna.

In [ ]:
flexion = viga.flexure_design
corte = viga.shear_design

print("Armadura inferior:", flexion.bottom)
print(f"  As adoptada  = {flexion.bottom.A_s.to('cm**2'):.2f~P}")
print(f"  As requerida = {flexion.bottom.A_s_req.to('cm**2'):.2f~P}")
print(f"  As mínima    = {flexion.bottom.A_s_min.to('cm**2'):.2f~P}")
print("Armadura superior:", flexion.top)
print("Estribos:", corte, f"({corte.n_legs} ramas)")
print(f"  Av adoptada  = {corte.A_v.to('cm**2/m'):.2f~P}")
print(f"  Av requerida = {corte.A_v_req.to('cm**2/m'):.2f~P}")
print(f"DCR flexión = {flexion.DCR:.2f} | DCR corte = {corte.DCR:.2f}")

**Memoria de cálculo detallada**

Para la combinación que gobierna, mento muestra el cálculo completo: propiedades de los
materiales, geometría, cada verificación del reglamento y la relación demanda-capacidad.

In [ ]:
nodo.flexure_results_detailed()

In [ ]:
nodo.shear_results_detailed()

**Exportar a Word y a Excel**

Las mismas memorias se exportan a un documento de Word, y las tablas a Excel. Los archivos se
escriben en el directorio de trabajo.

In [ ]:
nodo.flexure_results_detailed_doc()
nodo.shear_results_detailed_doc()

nodo.check_flexure().to_excel("CIRSOC 201-25 resultados flexion.xlsx")
nodo.check_shear().to_excel("CIRSOC 201-25 resultados corte.xlsx")